# Step 4A.5 — B2_M: sensitivity / upper-bound structured model
**НЕ temporally-safe historical forecast.** `cvss_score`, `cwe_ids`, `products` походять з
поточного NVD-знімка, і Step 4A.3 показав `last_modified > t_ref` для 92,07% CVE (100% для
позитивів H=180) — тому цей експеримент явно позначається як sensitivity/upper-bound,
а не як заміна `B2_S`. Той самий frozen спліт, той самий протокол (test — одноразово,
early stopping лише на validation), для прямого порівняння з `B0/B1/B2_S`.

Ядро: IDS Research.

In [17]:
import sys, subprocess
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True)
    import lightgbm as lgb

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import average_precision_score, roc_auc_score

CTI_DIR = Path("/srv/ids_research/data/cti")
RESULTS_DIR = Path.home() / "notebooks" / "2 article" / "release" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

HORIZON = 180
TARGET_COL = f"future_kev_{HORIZON}d"
SPLIT_COL = f"split_{HORIZON}d"
TOP_K_FRACTIONS = [0.001, 0.005, 0.01, 0.05]
RANDOM_STATE = 42

EXPECTED_SPLIT_SIZES = {"train": 249725, "val": 15192, "test": 65159}
print(f"HORIZON={HORIZON}, lightgbm {lgb.__version__}")

HORIZON=180, lightgbm 4.7.0


In [18]:
# ---------- Load + safe derived features (той самий код, що в step4a4) ----------
ds = pd.read_parquet(CTI_DIR / "cve_forecast_dataset.parquet")

def add_safe_derived_features(df):
    df = df.copy()
    df["cve_age_days"] = (pd.to_datetime(df["t_ref"]) - pd.to_datetime(df["published"])).dt.days
    return df

ds = add_safe_derived_features(ds)
ds["post_epss_published"] = ds["post_epss_published"].astype(int)

observed = ds[ds[TARGET_COL].notna()].copy()
observed[TARGET_COL] = observed[TARGET_COL].astype(int)

for split_name, expected in EXPECTED_SPLIT_SIZES.items():
    actual = len(observed[observed[SPLIT_COL] == split_name])
    assert actual == expected, f"{split_name}: actual={actual}, expected={expected} — dataset змінився!"
print("✅ Розміри train/val/test збігаються із зафіксованим Step 4A.2")

train = observed[observed[SPLIT_COL] == "train"].copy()
val = observed[observed[SPLIT_COL] == "val"].copy()
test = observed[observed[SPLIT_COL] == "test"].copy()
print(f"train={len(train):,}  val={len(val):,}  test={len(test):,}")

✅ Розміри train/val/test збігаються із зафіксованим Step 4A.2
train=249,725  val=15,192  test=65,159


In [19]:
# ---------- Аудит CWE / products: occurrence coverage ТА row coverage, K поки НЕ фіксуємо ----------
from collections import Counter

def flat_counter(series):
    c = Counter()
    for lst in series:
        if lst is not None and len(lst) > 0:
            c.update(lst)
    return c

cwe_counts = flat_counter(train["cwe_ids"])
prod_counts = flat_counter(train["products"])

def occurrence_coverage_for_k(counts, k):
    """Частка всіх ВХОДЖЕНЬ (не CVE-рядків) CWE/products, що припадає на top-K категорій."""
    top_k = {x for x, _ in counts.most_common(k)}
    covered = sum(v for x, v in counts.items() if x in top_k)
    total = sum(counts.values())
    return covered / total if total else 0.0

def row_coverage_for_k(series, counts, k):
    """Частка CVE-РЯДКІВ (серед тих, що мають хоча б один CWE/product), де є ХОЧА Б ОДИН
    елемент з top-K. Це змістовніший показник для вибору K, ніж occurrence coverage."""
    top_k = {x for x, _ in counts.most_common(k)}
    rows_with_any = 0
    rows_with_top_k = 0
    for lst in series:
        if lst is None or len(lst) == 0:
            continue
        rows_with_any += 1
        if any(x in top_k for x in lst):
            rows_with_top_k += 1
    return rows_with_top_k / rows_with_any if rows_with_any else 0.0

print(f"Унікальних CWE у train: {len(cwe_counts)}, унікальних products: {len(prod_counts)}")

print("\nCWE — occurrence coverage / row coverage для різних K:")
for k in [10, 20, 30, 50, 100]:
    occ = occurrence_coverage_for_k(cwe_counts, k)
    row = row_coverage_for_k(train["cwe_ids"], cwe_counts, k)
    print(f"  top-{k:3d}: occurrence={occ:.1%}, row={row:.1%}")

print("\nProducts — occurrence coverage / row coverage для різних K:")
for k in [10, 20, 30, 50, 100]:
    occ = occurrence_coverage_for_k(prod_counts, k)
    row = row_coverage_for_k(train["products"], prod_counts, k)
    print(f"  top-{k:3d}: occurrence={occ:.1%}, row={row:.1%}")

print("\n⚠ K НЕ фіксується тут автоматично. Перегляньте цифри вище і встановіть")
print("  CWE_TOP_K / PRODUCTS_TOP_K у НАСТУПНІЙ комірці вручну, виходячи з row coverage")
print("  (не occurrence coverage — вона завжди виглядає оптимістичніше) і БЕЗ огляду на")
print("  validation AP (K обирається за словниковим покриттям, а не tuning, щоб B2_M")
print("  залишався sensitivity-експериментом, а не прихованим підбором під val).")


Унікальних CWE у train: 679, унікальних products: 156415

CWE — occurrence coverage / row coverage для різних K:
  top- 10: occurrence=52.6%, row=56.0%
  top- 20: occurrence=68.1%, row=71.9%
  top- 30: occurrence=76.9%, row=79.9%
  top- 50: occurrence=86.2%, row=88.7%
  top-100: occurrence=94.6%, row=96.5%

Products — occurrence coverage / row coverage для різних K:
  top- 10: occurrence=5.0%, row=18.2%
  top- 20: occurrence=7.6%, row=20.7%
  top- 30: occurrence=9.0%, row=21.8%
  top- 50: occurrence=10.9%, row=23.8%
  top-100: occurrence=13.9%, row=25.6%

⚠ K НЕ фіксується тут автоматично. Перегляньте цифри вище і встановіть
  CWE_TOP_K / PRODUCTS_TOP_K у НАСТУПНІЙ комірці вручну, виходячи з row coverage
  (не occurrence coverage — вона завжди виглядає оптимістичніше) і БЕЗ огляду на
  validation AP (K обирається за словниковим покриттям, а не tuning, щоб B2_M
  залишався sensitivity-експериментом, а не прихованим підбором під val).


In [20]:
# ---------- ЯВНИЙ вибір K (за row coverage з Cell 3, вручну, БЕЗ огляду на val AP) ----------
# Встановіть за фактичними цифрами аудиту вище. Значення нижче — стартова точка,
# скоригуйте після перегляду row coverage для CWE і products.
CWE_TOP_K = 30
PRODUCTS_TOP_K = 30
TOP_CWES = [x for x, _ in cwe_counts.most_common(CWE_TOP_K)]
TOP_PRODUCTS = [x for x, _ in prod_counts.most_common(PRODUCTS_TOP_K)]
print(f"CWE_TOP_K={CWE_TOP_K} (row coverage={row_coverage_for_k(train['cwe_ids'], cwe_counts, CWE_TOP_K):.1%}), "
      f"PRODUCTS_TOP_K={PRODUCTS_TOP_K} (row coverage={row_coverage_for_k(train['products'], prod_counts, PRODUCTS_TOP_K):.1%})")

# ---------- Кодування B2_M: multi-hot(top-K) + missing + count(УНІКАЛЬНИХ елементів) ----------
def _clean_name(item):
    return str(item).replace("-", "_").replace(":", "_").replace(".", "_")

def encode_multihot(df, list_col, top_items, prefix):
    out = pd.DataFrame(index=df.index)
    for item in top_items:
        colname = f"{prefix}_{_clean_name(item)}"
        out[colname] = df[list_col].map(lambda lst: int(lst is not None and item in lst))
    out[f"{prefix}_missing"] = df[list_col].map(lambda lst: int(lst is None or len(lst) == 0))
    # count = КІЛЬКІСТЬ УНІКАЛЬНИХ елементів, не len(lst) — не покладаємось мовчки на
    # відсутність дублікатів у вихідних списках cwe_ids/products.
    out[f"{prefix}_count"] = df[list_col].map(lambda lst: len(set(lst)) if lst is not None else 0)
    return out

# Перевірка колізій назв feature-колонок ДО побудови матриці (напр. "abc:def" і "abc_def"
# після очищення могли б збігтися для products — для CWE ризик малий, але перевіряємо обидва).
cwe_feature_names = [f"cwe_{_clean_name(item)}" for item in TOP_CWES]
prod_feature_names = [f"prod_{_clean_name(item)}" for item in TOP_PRODUCTS]
assert len(cwe_feature_names) == len(set(cwe_feature_names)), "Колізія назв CWE features після очищення"
assert len(prod_feature_names) == len(set(prod_feature_names)), "Колізія назв product features після очищення"
print("✅ Колізій назв feature-колонок не знайдено")

def build_b2m_matrix(df):
    base = df[["epss_t", "cve_age_days", "post_epss_published"]].copy()
    base["cvss_score"] = df["cvss_score"]
    base["cvss_missing"] = df["cvss_score"].isna().astype(int)
    base["cvss_score"] = base["cvss_score"].fillna(0.0)   # missing позначено окремим прапорцем вище
    cwe_feats = encode_multihot(df, "cwe_ids", TOP_CWES, "cwe")
    prod_feats = encode_multihot(df, "products", TOP_PRODUCTS, "prod")
    return pd.concat([base, cwe_feats, prod_feats], axis=1)

X_train_m = build_b2m_matrix(train)
X_val_m = build_b2m_matrix(val)
X_test_m = build_b2m_matrix(test)
y_train = train[TARGET_COL]
y_val = val[TARGET_COL]
y_test = test[TARGET_COL]

B2_M_FEATURES = list(X_train_m.columns)
print(f"B2_M: {len(B2_M_FEATURES)} ознак (3 base + cvss×2 + cwe×{CWE_TOP_K+2} + prod×{PRODUCTS_TOP_K+2})")

for name, X in [("train", X_train_m), ("val", X_val_m), ("test", X_test_m)]:
    assert X.isna().sum().sum() == 0, f"NaN у X_{name} після кодування"
    assert np.isfinite(X.to_numpy()).all(), f"inf/-inf у X_{name}"
print("✅ NaN/inf-перевірки пройдено для train, val, test")


CWE_TOP_K=30 (row coverage=79.9%), PRODUCTS_TOP_K=30 (row coverage=21.8%)
✅ Колізій назв feature-колонок не знайдено
B2_M: 69 ознак (3 base + cvss×2 + cwe×32 + prod×32)
✅ NaN/inf-перевірки пройдено для train, val, test


In [21]:
# ---------- Train B2_M (той самий протокол, що й B2_S: scale_pos_weight з train, unweighted AP на val) ----------
n_pos_train = int(y_train.sum())
n_neg_train = len(y_train) - n_pos_train
scale_pos_weight = n_neg_train / n_pos_train
print(f"n_pos_train={n_pos_train}, scale_pos_weight={scale_pos_weight:.2f}")

def unweighted_average_precision(preds, dataset):
    y_true = dataset.get_label()
    return "AP_unweighted", average_precision_score(y_true, preds), True

lgb_train_m = lgb.Dataset(X_train_m, label=y_train)
lgb_val_m = lgb.Dataset(X_val_m, label=y_val, reference=lgb_train_m)

params_m = dict(
    objective="binary", metric="None", scale_pos_weight=scale_pos_weight,
    verbosity=-1, seed=RANDOM_STATE, num_leaves=15, learning_rate=0.05, min_data_in_leaf=50,
)
b2m_model = lgb.train(
    params_m, lgb_train_m, num_boost_round=500,
    valid_sets=[lgb_val_m], valid_names=["val"],
    feval=unweighted_average_precision,
    callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False), lgb.log_evaluation(period=0)],
)
print(f"B2_M: best_iteration={b2m_model.best_iteration} (за unweighted AP на VALIDATION, не на test)")

n_pos_train=366, scale_pos_weight=681.31
B2_M: best_iteration=1 (за unweighted AP на VALIDATION, не на test)


In [22]:
# ---------- Прогнози на val і test (test — ОДИН РАЗ, тут) ----------
b2m_val_scores = b2m_model.predict(X_val_m, num_iteration=b2m_model.best_iteration)
b2m_test_scores = b2m_model.predict(X_test_m, num_iteration=b2m_model.best_iteration)

ap_val = average_precision_score(y_val, b2m_val_scores)
ap_test = average_precision_score(y_test, b2m_test_scores)
roc_val = roc_auc_score(y_val, b2m_val_scores)
roc_test = roc_auc_score(y_test, b2m_test_scores)
print(f"B2_M val:  AP={ap_val:.6f}  ROC-AUC={roc_val:.4f}")
print(f"B2_M test: AP={ap_test:.6f}  ROC-AUC={roc_test:.4f}")

B2_M val:  AP=0.033078  ROC-AUC=0.7322
B2_M test: AP=0.011209  ROC-AUC=0.7267


In [23]:
# ---------- Precision/Recall@K для B2_M (test) + порівняння з замороженими B0/B1/B2_S ----------
def precision_recall_at_k(y_true, y_score, k_fraction):
    n = len(y_true)
    k = max(1, int(round(n * k_fraction)))
    order = np.argsort(-y_score)
    top_k_idx = order[:k]
    n_pos_in_top_k = y_true.values[top_k_idx].sum()
    n_pos_total = y_true.sum()
    return n_pos_in_top_k / k, (n_pos_in_top_k / n_pos_total if n_pos_total else np.nan), k

topk_rows_m = []
for frac in TOP_K_FRACTIONS:
    p, r, k = precision_recall_at_k(y_test, b2m_test_scores, frac)
    topk_rows_m.append(dict(top_pct=f"{frac:.1%}", K=k, precision=p, recall=r))
topk_m_df = pd.DataFrame(topk_rows_m)
print("B2_M Precision/Recall@K (test):")
display(topk_m_df)

# Порівняння з B0/B1/B2_S зі Step 4A.4 (якщо звіт уже збережено на диску)
prev_summary_path = RESULTS_DIR / f"step4a4_baseline_summary_h{HORIZON}.csv"
if prev_summary_path.exists():
    prev = pd.read_csv(prev_summary_path, index_col=0)
    b2m_row = dict(ap=ap_test, roc_auc=roc_test)
    for frac in TOP_K_FRACTIONS:
        row = topk_m_df[topk_m_df["top_pct"] == f"{frac:.1%}"].iloc[0]
        b2m_row[f"precision@{frac:.1%}"] = row["precision"]
        b2m_row[f"recall@{frac:.1%}"] = row["recall"]
    combined = pd.concat([prev, pd.DataFrame([b2m_row], index=["B2_M (sensitivity/upper-bound)"])])
    print("\n" + "=" * 70)
    print("ПОРІВНЯННЯ: B0 / B1 / B2_S (temporally-safe, frozen) vs B2_M (sensitivity/upper-bound)")
    print("=" * 70)
    display(combined)
else:
    print(f"\n⚠ {prev_summary_path} не знайдено — запустіть step4a4_baseline_models.ipynb до кінця спочатку")

B2_M Precision/Recall@K (test):


,top_pct,K,precision,recall
0,0.1%,65,0.030769,0.022222
1,0.5%,326,0.042945,0.155556
2,1.0%,652,0.027607,0.200000
3,5.0%,3258,0.009515,0.344444



ПОРІВНЯННЯ: B0 / B1 / B2_S (temporally-safe, frozen) vs B2_M (sensitivity/upper-bound)


,ap,roc_auc,precision@0.1%,recall@0.1%,precision@0.5%,recall@0.5%,precision@1.0%,recall@1.0%,precision@5.0%,recall@5.0%
B0,0.001381,0.500000,0.001381,0.000998,0.001381,0.005003,0.001381,0.010006,0.001381,0.050001
B1,0.018597,0.715477,0.046154,0.033333,0.042945,0.155556,0.024540,0.177778,0.009515,0.344444
B2_S,0.007880,0.573759,0.061538,0.044444,0.033742,0.122222,0.018405,0.133333,0.005525,0.200000
B2_M (sensitivity/upper-bound),0.011209,0.726713,0.030769,0.022222,0.042945,0.155556,0.027607,0.200000,0.009515,0.344444


In [24]:
# ---------- Feature importance B2_M (топ-15) — чи CVSS/CWE/products реально впливають ----------
importance_m = pd.Series(b2m_model.feature_importance(importance_type="gain"), index=B2_M_FEATURES)
importance_m_pct = 100 * importance_m / importance_m.sum()
print("Топ-15 ознак за gain (B2_M):")
display(pd.DataFrame({"gain": importance_m, "gain_pct": importance_m_pct}).sort_values("gain", ascending=False).head(15))

Топ-15 ознак за gain (B2_M):


,gain,gain_pct
cvss_score,24724108.0,43.883461
epss_t,10531110.0,18.691940
cve_age_days,8343135.0,14.808447
post_epss_published,7098860.0,12.599951
prod_count,4959290.0,8.802372
prod_debian_debian_linux,683876.0,1.213829
cvss_missing,0.0,0.000000
cwe_CWE_79,0.0,0.000000
cwe_CWE_119,0.0,0.000000
cwe_CWE_20,0.0,0.000000


In [25]:
# ---------- ОБОВ'ЯЗКОВЕ маркування результату + збереження ----------
print("=" * 70)
print("⚠ B2_M Є SENSITIVITY / UPPER-BOUND-STYLE МОДЕЛЛЮ, НЕ temporally-safe historical forecast.")
print("  cvss_score/cwe_ids/products походять з ПОТОЧНОГО знімка NVD; Step 4A.3 показав")
print("  last_modified > t_ref для 92.07% CVE (100% для 518 позитивів H=180). Це оцінка")
print("  ПОТЕНЦІЙНОГО внеску структурованих NVD-метаданих за умови їхньої доступності")
print("  на момент t_ref — не математично доведена верхня межа якості моделі.")
print("=" * 70)

out_row = dict(model="B2_M_sensitivity_upper_bound", horizon=HORIZON, ap_test=ap_test, roc_auc_test=roc_test,
               ap_val=ap_val, roc_auc_val=roc_val, best_iteration=b2m_model.best_iteration,
               n_features=len(B2_M_FEATURES), cwe_top_k=CWE_TOP_K, products_top_k=PRODUCTS_TOP_K,
               scale_pos_weight=scale_pos_weight)
pd.DataFrame([out_row]).to_csv(RESULTS_DIR / f"step4a5_b2m_summary_h{HORIZON}.csv", index=False)
topk_m_df.to_csv(RESULTS_DIR / f"step4a5_b2m_topk_h{HORIZON}.csv", index=False)
b2m_model.save_model(str(RESULTS_DIR / f"step4a5_b2m_model_h{HORIZON}.txt"))
print(f"\nЗбережено в {RESULTS_DIR}")

⚠ B2_M Є SENSITIVITY / UPPER-BOUND-STYLE МОДЕЛЛЮ, НЕ temporally-safe historical forecast.
  cvss_score/cwe_ids/products походять з ПОТОЧНОГО знімка NVD; Step 4A.3 показав
  last_modified > t_ref для 92.07% CVE (100% для 518 позитивів H=180). Це оцінка
  ПОТЕНЦІЙНОГО внеску структурованих NVD-метаданих за умови їхньої доступності
  на момент t_ref — не математично доведена верхня межа якості моделі.

Збережено в /home/ivan_r_opirskyy/notebooks/2 article/release/results
